In [1]:
# ============================================================
# PRACTICA 1 - VISUALIZACIÓN DE GRUPOS EN DIMENSIÓN REDUCIDA
# ÍNDICE DE MARGINACIÓN MUNICIPAL
# ------------------------------------------------------------
# Autor: Bryan Eduardo Martínez Cortes
# Diplomado Ciencia de Datos - Generación 33
# ------------------------------------------------------------
# Descripción:
# Script profesional para:
# 1) Cargar el dataset ITER 2020 desde archivo ZIP
# 2) Filtrar el nivel municipal
# 3) Realizar EDA breve
# 4) Construir variables relativas asociadas a marginación
# 5) Reducir dimensionalidad a 2 componentes
# 6) Crear un índice sintético de marginación con PCA
# 7) Exportar resultados y gráficas
# ============================================================

from __future__ import annotations

import os
import zipfile
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")



In [2]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

ZIP_PATH: str = "M3P1_Metodos_Reduccion.zip"
OUTPUT_DIR: str = "outputs_m3_p1"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ============================================================
# UTILIDADES
# ============================================================

@dataclass
class ReductionArtifacts:
    """Contenedor de resultados del proceso de reducción."""
    df_resultado: pd.DataFrame
    df_variables: pd.DataFrame
    df_pesos: pd.DataFrame
    explained_variance_ratio: np.ndarray


def log(message: str) -> None:
    """Imprime mensajes de seguimiento del proceso."""
    print(f"[INFO] {message}")


def ensure_bom_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Corrige nombres de columnas con BOM o caracteres extraños.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame original.

    Returns
    -------
    pd.DataFrame
        DataFrame con nombres corregidos.
    """
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").replace("ï»¿", "") for c in df.columns]
    return df


In [4]:
def load_csv_from_zip(zip_path: str, inner_csv_keyword: str) -> pd.DataFrame:
    """
    Carga un CSV contenido dentro de un ZIP.

    Parameters
    ----------
    zip_path : str
        Ruta del archivo ZIP.
    inner_csv_keyword : str
        Parte del nombre del archivo interno a localizar.

    Returns
    -------
    pd.DataFrame
        DataFrame cargado.
    """
    with zipfile.ZipFile(zip_path, "r") as zf:
        candidates = [
            name for name in zf.namelist()
            if name.lower().endswith(".csv") and inner_csv_keyword.lower() in name.lower()
        ]

        if not candidates:
            raise FileNotFoundError(
                f"No se encontró un CSV dentro del ZIP que contenga: {inner_csv_keyword}"
            )

        selected_file = candidates[0]
        log(f"Archivo encontrado dentro del ZIP: {selected_file}")

        with zf.open(selected_file) as f:
            df = pd.read_csv(f, encoding="latin-1", low_memory=False)

    df = ensure_bom_columns(df)
    return df

In [5]:
def save_dataframe(df: pd.DataFrame, filename: str) -> None:
    """
    Guarda un DataFrame en CSV dentro de OUTPUT_DIR.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a guardar.
    filename : str
        Nombre del archivo CSV.
    """
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    log(f"Archivo guardado: {path}")


def save_figure(filename: str) -> None:
    """
    Guarda la figura actual de matplotlib en OUTPUT_DIR.

    Parameters
    ----------
    filename : str
        Nombre del archivo de imagen.
    """
    path = os.path.join(OUTPUT_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    log(f"Gráfica guardada: {path}")

In [6]:
def clean_numeric_columns(df: pd.DataFrame, exclude_cols: List[str]) -> pd.DataFrame:
    """
    Convierte a numérico todas las columnas excepto las excluidas.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame original.
    exclude_cols : List[str]
        Columnas que no deben convertirse.

    Returns
    -------
    pd.DataFrame
        DataFrame con columnas numéricas convertidas.
    """
    df = df.copy()
    for col in df.columns:
        if col not in exclude_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """
    Divide dos series de forma segura evitando divisiones entre cero.

    Parameters
    ----------
    numerator : pd.Series
        Numerador.
    denominator : pd.Series
        Denominador.

    Returns
    -------
    pd.Series
        Resultado de la división segura.
    """
    result = numerator / denominator.replace({0: np.nan})
    return result.replace([np.inf, -np.inf], np.nan)

In [7]:
# ============================================================
# CARGA Y PREPARACIÓN
# ============================================================

def load_municipal_data(zip_path: str) -> pd.DataFrame:
    """
    Carga el dataset de conjunto de datos y filtra el nivel municipal.

    Reglas de filtrado:
    - LOC == '0000' corresponde a total municipal.
    - Se excluyen registros agregados nacionales/estatales cuando MUN == '000'.

    Parameters
    ----------
    zip_path : str
        Ruta del ZIP.

    Returns
    -------
    pd.DataFrame
        DataFrame a nivel municipio.
    """
    df = load_csv_from_zip(zip_path, "conjunto_de_datos")
    log(f"Dimensión original del dataset: {df.shape}")

    id_cols = ["ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC"]
    for col in ["ENTIDAD", "MUN", "LOC"]:
        df[col] = df[col].astype(str).str.zfill(3 if col != "LOC" else 4)

    df = clean_numeric_columns(
        df=df,
        exclude_cols=[
            "ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC", "LONGITUD", "LATITUD"
        ]
    )

    # Filtrar totales municipales
    df_mun = df.loc[(df["LOC"] == "0000") & (df["MUN"] != "000")].copy()

    # Excluir posibles filas especiales
    df_mun = df_mun[~df_mun["NOM_LOC"].astype(str).str.contains("vivienda", case=False, na=False)]

    log(f"Dimensión a nivel municipio: {df_mun.shape}")

    # Crear llave municipal
    df_mun["CVEGEO"] = df_mun["ENTIDAD"].str.zfill(2) + df_mun["MUN"].str.zfill(3)

    # Reordenar columnas principales
    ordered_cols = ["CVEGEO", "ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC"]
    other_cols = [c for c in df_mun.columns if c not in ordered_cols]
    df_mun = df_mun[ordered_cols + other_cols]

    return df_mun



In [8]:
def run_brief_eda(df: pd.DataFrame) -> None:
    """
    Ejecuta un EDA breve y guarda resultados básicos.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame municipal.
    """
    log("Iniciando EDA breve...")

    summary = pd.DataFrame({
        "columna": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
        "n_unicos": df.nunique(dropna=True).values
    })
    save_dataframe(summary, "eda_resumen_estructura.csv")

    # Top faltantes
    missing = (
        df.isna().mean()
        .sort_values(ascending=False)
        .mul(100)
        .rename("pct_nulos")
        .reset_index()
        .rename(columns={"index": "variable"})
    )
    save_dataframe(missing, "eda_faltantes.csv")

    top_missing = missing.head(20).sort_values("pct_nulos", ascending=True)

    plt.figure(figsize=(10, 7))
    plt.barh(top_missing["variable"], top_missing["pct_nulos"])
    plt.title("Top 20 variables con mayor porcentaje de faltantes")
    plt.xlabel("% de valores faltantes")
    plt.ylabel("Variable")
    save_figure("eda_top20_faltantes.png")

    # Estadísticos de variables clave si existen
    key_candidates = ["POBTOT", "TVIVHAB", "PROM_OCUP", "PRO_OCUP_C", "GRAPROES"]
    key_present = [c for c in key_candidates if c in df.columns]

    if key_present:
        desc = df[key_present].describe().T
        desc = desc.reset_index().rename(columns={"index": "variable"})
        save_dataframe(desc, "eda_estadisticos_clave.csv")

In [9]:
def build_margination_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """
    Construye variables relativas para aproximar marginación municipal.

    Se privilegian tasas/proporciones en lugar de conteos absolutos para
    hacer comparables municipios de distinto tamaño.
    """
    work = df.copy()

    # ------------------------------------------------------------
    # Resolver nombre correcto de población de 15 años y más
    # ------------------------------------------------------------
    if "P15YMAS" in work.columns:
        col_p15ymas = "P15YMAS"
    elif "P_15YMAS" in work.columns:
        col_p15ymas = "P_15YMAS"
    else:
        raise ValueError(
            "No se encontró la columna de población de 15 años y más "
            "('P15YMAS' ni 'P_15YMAS')."
        )

    # ------------------------------------------------------------
    # Validación de columnas requeridas
    # ------------------------------------------------------------
    required_cols = [
        "P15YM_AN",
        "P15YM_SE",
        "PSINDER",
        "POBTOT",
        "VPH_PISOTI",
        "VPH_S_ELEC",
        "VPH_AGUAFV",
        "VPH_NODREN",
        "TVIVHAB",
        "PRO_OCUP_C",
        "VPH_INTER"
    ]

    missing_required = [c for c in required_cols if c not in work.columns]
    if missing_required:
        raise ValueError(
            f"Faltan columnas necesarias para el índice: {missing_required}"
        )

    # ------------------------------------------------------------
    # Variables relativas de marginación
    # ------------------------------------------------------------
    work["pct_analfabetismo_15ymas"] = safe_divide(work["P15YM_AN"], work[col_p15ymas])
    work["pct_sin_escolaridad_15ymas"] = safe_divide(work["P15YM_SE"], work[col_p15ymas])
    work["pct_sin_derechohabiencia"] = safe_divide(work["PSINDER"], work["POBTOT"])
    work["pct_piso_tierra"] = safe_divide(work["VPH_PISOTI"], work["TVIVHAB"])
    work["pct_sin_electricidad"] = safe_divide(work["VPH_S_ELEC"], work["TVIVHAB"])
    work["pct_sin_agua_entubada"] = safe_divide(work["VPH_AGUAFV"], work["TVIVHAB"])
    work["pct_sin_drenaje"] = safe_divide(work["VPH_NODREN"], work["TVIVHAB"])
    work["prom_ocupantes_por_cuarto"] = work["PRO_OCUP_C"]
    work["pct_sin_internet"] = 1 - safe_divide(work["VPH_INTER"], work["TVIVHAB"])

    feature_justification = {
        "pct_analfabetismo_15ymas": "Captura rezago educativo severo en población de 15 años y más.",
        "pct_sin_escolaridad_15ymas": "Mide exclusión educativa estructural.",
        "pct_sin_derechohabiencia": "Refleja carencia de acceso a servicios de salud.",
        "pct_piso_tierra": "Proxy clásico de precariedad en vivienda.",
        "pct_sin_electricidad": "Indica rezago en infraestructura básica del hogar.",
        "pct_sin_agua_entubada": "Mide carencia de acceso a agua en la vivienda.",
        "pct_sin_drenaje": "Refleja rezago sanitario e infraestructura deficiente.",
        "prom_ocupantes_por_cuarto": "Aproxima hacinamiento habitacional.",
        "pct_sin_internet": "Representa rezago en conectividad y acceso a información."
    }

    selected_features = list(feature_justification.keys())

    ratio_features = [f for f in selected_features if f != "prom_ocupantes_por_cuarto"]
    for col in ratio_features:
        work[col] = work[col].clip(0, 1)

    df_variables = pd.DataFrame({
        "variable": selected_features,
        "justificacion": [feature_justification[v] for v in selected_features]
    })
    save_dataframe(df_variables, "resumen_variables_seleccionadas.csv")

    return work, feature_justification

In [10]:
# ============================================================
# REDUCCIÓN DE DIMENSIONALIDAD E ÍNDICE
# ============================================================

def build_margination_index(df: pd.DataFrame, feature_names: List[str]) -> ReductionArtifacts:
    """
    Construye un índice sintético de marginación usando PCA.

    Flujo:
    - Imputación mediana
    - Estandarización
    - PCA para índice (PC1)
    - PCA para visualización 2D (PC1, PC2)

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame municipal con variables derivadas.
    feature_names : List[str]
        Variables seleccionadas para el índice.

    Returns
    -------
    ReductionArtifacts
        Resultados del proceso.
    """
    model_input = df[feature_names].copy()

    pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=2, random_state=42))
    ])

    X_pca = pipeline.fit_transform(model_input)
    pca_model: PCA = pipeline.named_steps["pca"]

    result = df[["CVEGEO", "NOM_ENT", "NOM_MUN", "POBTOT"]].copy()
    result["PC1"] = X_pca[:, 0]
    result["PC2"] = X_pca[:, 1]

    # Índice de marginación:
    # Se usa PC1 y se orienta para que valores mayores = mayor marginación
    # La orientación depende del signo de las cargas.
    loadings = pd.DataFrame(
        pca_model.components_.T,
        index=feature_names,
        columns=["PC1", "PC2"]
    ).reset_index().rename(columns={"index": "variable"})

    # Si la suma de cargas de variables de privación sale negativa, invertimos signo
    deprivation_sum = loadings["PC1"].sum()
    sign = -1 if deprivation_sum < 0 else 1

    result["indice_marginacion"] = result["PC1"] * sign
    result["PC1_visual"] = result["PC1"] * sign
    loadings["PC1"] = loadings["PC1"] * sign

    # Estandarización del índice a media 0, desv 1
    result["indice_marginacion_z"] = (
        (result["indice_marginacion"] - result["indice_marginacion"].mean()) /
        result["indice_marginacion"].std(ddof=0)
    )

    # Escala 0-100
    min_idx = result["indice_marginacion"].min()
    max_idx = result["indice_marginacion"].max()
    result["indice_marginacion_0_100"] = (
        100 * (result["indice_marginacion"] - min_idx) / (max_idx - min_idx)
    )

    # Cuantiles interpretables
    result["grupo_marginacion_quintil"] = pd.qcut(
        result["indice_marginacion_0_100"],
        q=5,
        labels=[
            "Muy baja",
            "Baja",
            "Media",
            "Alta",
            "Muy alta"
        ]
    )

    explained = pca_model.explained_variance_ratio_

    return ReductionArtifacts(
        df_resultado=result.sort_values("indice_marginacion_0_100", ascending=False).reset_index(drop=True),
        df_variables=df[feature_names].copy(),
        df_pesos=loadings.copy(),
        explained_variance_ratio=explained
    )



In [11]:
# ============================================================
# VISUALIZACIONES
# ============================================================

def plot_correlation_matrix(df: pd.DataFrame, feature_names: List[str]) -> None:
    """
    Genera una matriz de correlación simple.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con variables del índice.
    feature_names : List[str]
        Variables seleccionadas.
    """
    corr = df[feature_names].corr()

    plt.figure(figsize=(10, 8))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(feature_names)), feature_names, rotation=90)
    plt.yticks(range(len(feature_names)), feature_names)
    plt.title("Matriz de correlación de variables seleccionadas")
    save_figure("correlacion_variables_marginacion.png")

    save_dataframe(
        corr.reset_index().rename(columns={"index": "variable"}),
        "correlacion_variables_marginacion.csv"
    )


def plot_pca_projection(df_resultado: pd.DataFrame) -> None:
    """
    Grafica la proyección 2D de municipios usando PC1 y PC2.

    Parameters
    ----------
    df_resultado : pd.DataFrame
        Resultado con componentes principales.
    """
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(
        df_resultado["PC1_visual"],
        df_resultado["PC2"],
        c=df_resultado["indice_marginacion_0_100"],
        alpha=0.75
    )
    plt.colorbar(scatter, label="Índice de marginación (0-100)")
    plt.xlabel("Componente principal 1")
    plt.ylabel("Componente principal 2")
    plt.title("Proyección 2D de municipios mediante PCA")
    save_figure("pca_2d_municipios.png")


def plot_top_bottom_municipios(df_resultado: pd.DataFrame, n: int = 15) -> None:
    """
    Grafica municipios con mayor y menor marginación.

    Parameters
    ----------
    df_resultado : pd.DataFrame
        DataFrame final con índice.
    n : int, optional
        Número de municipios por extremo, by default 15
    """
    top = df_resultado.nlargest(n, "indice_marginacion_0_100").copy()
    bottom = df_resultado.nsmallest(n, "indice_marginacion_0_100").copy()

    plt.figure(figsize=(11, 7))
    plt.barh(top["NOM_MUN"] + ", " + top["NOM_ENT"], top["indice_marginacion_0_100"])
    plt.title(f"Top {n} municipios con mayor marginación")
    plt.xlabel("Índice de marginación (0-100)")
    plt.ylabel("Municipio")
    save_figure("top_municipios_mayor_marginacion.png")

    plt.figure(figsize=(11, 7))
    plt.barh(bottom["NOM_MUN"] + ", " + bottom["NOM_ENT"], bottom["indice_marginacion_0_100"])
    plt.title(f"Top {n} municipios con menor marginación")
    plt.xlabel("Índice de marginación (0-100)")
    plt.ylabel("Municipio")
    save_figure("top_municipios_menor_marginacion.png")


In [12]:
# ============================================================
# REPORTE TABULAR
# ============================================================

def generate_summary_outputs(
    artifacts: ReductionArtifacts,
    feature_justification: Dict[str, str]
) -> None:
    """
    Genera archivos de salida tabulares del modelo.

    Parameters
    ----------
    artifacts : ReductionArtifacts
        Resultados del proceso.
    feature_justification : Dict[str, str]
        Justificación de las variables.
    """
    save_dataframe(artifacts.df_resultado, "municipios_indice_marginacion.csv")

    pesos = artifacts.df_pesos.copy()
    pesos["abs_PC1"] = pesos["PC1"].abs()
    pesos = pesos.sort_values("abs_PC1", ascending=False)
    save_dataframe(pesos, "pesos_pca_indice.csv")

    explained = pd.DataFrame({
        "componente": ["PC1", "PC2"],
        "varianza_explicada": artifacts.explained_variance_ratio
    })
    save_dataframe(explained, "varianza_explicada_pca.csv")

    resumen = pd.DataFrame({
        "variable": list(feature_justification.keys()),
        "justificacion": list(feature_justification.values())
    }).merge(
        pesos[["variable", "PC1", "PC2"]],
        on="variable",
        how="left"
    )
    save_dataframe(resumen, "resumen_variables_justificacion_y_pesos.csv")

    # Top y bottom municipios
    save_dataframe(
        artifacts.df_resultado.nlargest(30, "indice_marginacion_0_100"),
        "top_30_municipios_mayor_marginacion.csv"
    )
    save_dataframe(
        artifacts.df_resultado.nsmallest(30, "indice_marginacion_0_100"),
        "top_30_municipios_menor_marginacion.csv"
    )



In [13]:
df_municipios = load_municipal_data(ZIP_PATH)
print("P_15YMAS" in df_municipios.columns)

[INFO] Archivo encontrado dentro del ZIP: iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv
[INFO] Dimensión original del dataset: (195662, 286)
[INFO] Dimensión a nivel municipio: (2469, 286)
True


In [14]:
# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================

def main() -> None:
    """Ejecuta el flujo completo de la práctica."""
    log("===== INICIO DEL PROCESO =====")

    # 1) Carga
    df_municipios = load_municipal_data(ZIP_PATH)

    # 2) EDA breve
    run_brief_eda(df_municipios)

    # 3) Variables de marginación
    df_features, feature_justification = build_margination_features(df_municipios)
    selected_features = list(feature_justification.keys())

    # 4) Reducción de dimensionalidad e índice
    artifacts = build_margination_index(df_features, selected_features)

    # 5) Visualizaciones
    plot_correlation_matrix(df_features, selected_features)
    plot_pca_projection(artifacts.df_resultado)
    plot_top_bottom_municipios(artifacts.df_resultado, n=15)

    # 6) Exportables
    generate_summary_outputs(artifacts, feature_justification)

    # 7) Resumen en consola
    log("===== RESUMEN DEL MODELO =====")
    print("\nVarianza explicada por los dos primeros componentes:")
    for i, ratio in enumerate(artifacts.explained_variance_ratio, start=1):
        print(f"  - PC{i}: {ratio:.4%}")

    print("\nTop 10 municipios con mayor marginación:")
    print(
        artifacts.df_resultado[
            ["CVEGEO", "NOM_ENT", "NOM_MUN", "indice_marginacion_0_100", "grupo_marginacion_quintil"]
        ].head(10).to_string(index=False)
    )

    print("\nTop 10 municipios con menor marginación:")
    print(
        artifacts.df_resultado.sort_values("indice_marginacion_0_100", ascending=True)[
            ["CVEGEO", "NOM_ENT", "NOM_MUN", "indice_marginacion_0_100", "grupo_marginacion_quintil"]
        ].head(10).to_string(index=False)
    )

    log("Proceso finalizado correctamente.")
    log(f"Revisa la carpeta de salidas: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

[INFO] ===== INICIO DEL PROCESO =====
[INFO] Archivo encontrado dentro del ZIP: iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv
[INFO] Dimensión original del dataset: (195662, 286)
[INFO] Dimensión a nivel municipio: (2469, 286)
[INFO] Iniciando EDA breve...
[INFO] Archivo guardado: outputs_m3_p1/eda_resumen_estructura.csv
[INFO] Archivo guardado: outputs_m3_p1/eda_faltantes.csv
[INFO] Gráfica guardada: outputs_m3_p1/eda_top20_faltantes.png
[INFO] Archivo guardado: outputs_m3_p1/eda_estadisticos_clave.csv
[INFO] Archivo guardado: outputs_m3_p1/resumen_variables_seleccionadas.csv
[INFO] Gráfica guardada: outputs_m3_p1/correlacion_variables_marginacion.png
[INFO] Archivo guardado: outputs_m3_p1/correlacion_variables_marginacion.csv
[INFO] Gráfica guardada: outputs_m3_p1/pca_2d_municipios.png
[INFO] Gráfica guardada: outputs_m3_p1/top_municipios_mayor_marginacion.png
[INFO] Gráfica guardada: outputs_m3_p1/top_municipios_menor_marginacion.png
[INFO] Archivo guardado: o